# Feature Engineering & TF-IDF Vectorization

## Objective

This notebook prepares the cleaned news text for machine learning.

The main steps are:

1. Load the preprocessed dataset
2. Define the target variable
3. Select the textual features
4. Split the dataset into training and testing sets
5. Convert text into numerical features using TF-IDF
6. Inspect the resulting feature matrices

The initial model will use only textual content and will exclude
subject and date-derived variables because exploratory analysis
revealed strong relationships between these variables and the target.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv("/Users/EXOTISCH TECH/Desktop/Fake-Real-News-Project/data/processed/news_cleaned.csv")

In [3]:
print("shape", df.shape)
df.head()

shape (44689, 7)


,title,text,subject,date,label,article_content,clean_text
0,It’s Really Happening: Trump Adviser Lays Out...,"Well, that didn t take long. In the short time...",News,2016-11-16,1,It’s Really Happening: Trump Adviser Lays Out...,it’s really happening trump adviser lay plan n...
1,Republican attempt to deflect Trump-Russia pro...,(Reuters) - Republican lawmaker Devin Nunes’ i...,politicsNews,2017-09-11,0,Republican attempt to deflect Trump-Russia pro...,republican attempt deflect trumprussia probe c...
2,Trump says churches should get FEMA funds for ...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,2017-09-09,0,Trump says churches should get FEMA funds for ...,trump say church get fema fund hurricane relie...
3,Trump Loves To Say The New York Times Is ‘Fai...,Print journalism and longstanding papers have ...,News,2017-08-07,1,Trump Loves To Say The New York Times Is ‘Fai...,trump love say new york time ‘failing’ – effor...
4,House Speaker Ryan briefed Trump on healthcare...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,2017-03-24,0,House Speaker Ryan briefed Trump on healthcare...,house speaker ryan briefed trump healthcare bi...


In [4]:
df.columns

Index(['title', 'text', 'subject', 'date', 'label', 'article_content',
       'clean_text'],
      dtype='str')

In [5]:
df.isnull().sum()

title               0
text                0
subject             0
date               10
label               0
article_content     0
clean_text          9
dtype: int64

In [6]:
df = df.dropna(subset=["clean_text"]).copy()


In [7]:
# rechheck
df.isnull().sum()

title              0
text               0
subject            0
date               1
label              0
article_content    0
clean_text         0
dtype: int64

In [8]:
X = df["clean_text"]
y = df["label"]

In [9]:
# check the split
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (44680,)
y shape: (44680,)


In [10]:
y.value_counts()

label
1    23469
0    21211
Name: count, dtype: int64

In [11]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
# check the split
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 35744
Testing samples: 8936


In [13]:
print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True))


Training distribution:
label
1    0.525263
0    0.474737
Name: proportion, dtype: float64

Testing distribution:
label
1    0.525291
0    0.474709
Name: proportion, dtype: float64


In [14]:
# TF_DF
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [15]:
# fit the vectorizer to the taining data
X_train_tfidf = tfidf.fit_transform(X_train)

In [16]:
X_test_tfidf = tfidf.transform(X_test)

In [17]:
# inspect the matrices
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (35744, 50000)
Testing TF-IDF shape: (8936, 50000)


In [18]:
# inspect vocabularly
feature_names = tfidf.get_feature_names_out()

print("Number of features:", len(feature_names))

Number of features: 50000


In [19]:
feature_names[:50]

array(['aa', 'aadhaar', 'aaplo', 'aaron', 'aarp', 'ab', 'aba', 'ababa',
       'aback', 'abadi', 'abadi said', 'abandon', 'abandoned',
       'abandoning', 'abbas', 'abbasi', 'abbott', 'abby', 'abc',
       'abc cbs', 'abc good', 'abc nbc', 'abc news', 'abc this',
       'abc week', 'abcpolitics', 'abdel', 'abdel fattah', 'abdication',
       'abdomen', 'abdrabbu', 'abdrabbu mansour', 'abducted', 'abduction',
       'abdul', 'abdulaziz', 'abdullah', 'abdullah saleh', 'abe',
       'abe said', 'abe told', 'abedi', 'abedin', 'abedini', 'abetting',
       'abhorrent', 'abid', 'abide', 'abiding', 'abidjan'], dtype=object)

In [23]:
# Actual TF_IDF values
X_train_tfidf[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 172 stored elements and shape (1, 50000)>

In [24]:
row = X_train_tfidf[0]

indices = row.nonzero()[1]

terms = [
    (feature_names[i], row[0, i])
    for i in indices
]

sorted(terms, key=lambda x: x[1], reverse=True)[:20]


[('refugee child', np.float64(0.23039583961379714)),
 ('stockholm', np.float64(0.16344694850599467)),
 ('beat', np.float64(0.1359013752828413)),
 ('blaming', np.float64(0.11992051373101485)),
 ('europe', np.float64(0.11528894887281955)),
 ('nationalist', np.float64(0.11524435745507156)),
 ('refugee', np.float64(0.11291242749789936)),
 ('mainstream press', np.float64(0.11098112899349104)),
 ('supporter attacked', np.float64(0.11036939177796588)),
 ('migrant', np.float64(0.11012252072573228)),
 ('ethnic religious', np.float64(0.10978485204621502)),
 ('trope', np.float64(0.10922519402239335)),
 ('roaming', np.float64(0.10817263412001502)),
 ('unspeakable', np.float64(0.10817263412001502)),
 ('something similar', np.float64(0.10817263412001502)),
 ('ran away', np.float64(0.10719811586341475)),
 ('every night', np.float64(0.10719811586341475)),
 ('traveling united', np.float64(0.10719811586341475)),
 ('child', np.float64(0.1060655443904901)),
 ('slowdown', np.float64(0.10503753423575468))]

In [26]:
import joblib

joblib.dump(tfidf, "/Users/EXOTISCH TECH/Desktop/Fake-Real-News-Project/models/tfidf_vectorizer.pkl")

['/Users/EXOTISCH TECH/Desktop/Fake-Real-News-Project/models/tfidf_vectorizer.pkl']

In [32]:
df.to_csv("/Users/EXOTISCH TECH/Desktop/Fake-Real-News-Project/data/processed/further_cleaned_news.csv")

# Summary

The cleaned news articles were converted into numerical features using
TF-IDF vectorization.

### Dataset preparation

- Target variable: `label`
- Text feature: `clean_text`
- Training/test split: 80/20
- Stratified splitting was used to preserve class proportions.

### TF-IDF configuration

- Maximum features: 50,000
- N-grams: unigrams and bigrams
- Minimum document frequency: 2
- Maximum document frequency: 95%
- Sublinear TF scaling: enabled

The TF-IDF vectorizer was fitted exclusively on the training data and
then used to transform the test data.

The resulting sparse feature matrices are ready for machine-learning
models.

The TF-IDF vectorizer was saved to the `models/` directory for future
use during model inference and deployment.

In [31]:
print("shape of df", df.shape)
print("shape of x_train_tfidf", X_train_tfidf.shape)
print("shape of x_test_tfidf", X_test_tfidf.shape)

shape of df (44680, 7)
shape of x_train_tfidf (35744, 50000)
shape of x_test_tfidf (8936, 50000)


In [30]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

label
1    0.525291
0    0.474709
Name: proportion, dtype: float64